manojcloudvm@instance-20260404-024439:~$ ip link
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN mode DEFAULT group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
2: ens4: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1460 qdisc mq state UP mode DEFAULT group default qlen 1000
    link/ether 42:01:0a:80:00:02 brd ff:ff:ff:ff:ff:ff
    altname enp0s4

https://www.youtube.com/watch?v=SSLpvcIOPK0

ip addr


Loopback (lo): A virtual interface that allows the system to communicate with itself, commonly known as localhost. It is assigned the address 127.0.0.1 in IPv4 or ::1 in IPv6
.
Ethernet (eth0 / enp...): The primary interface for physical wired network connections. It handles data transmission over a physical cable and can be monitored for a "link beat" to verify a connection
.
Wireless (wlan0): The interface used for Wi-Fi and wireless network connections. These interfaces are managed using specific tools like iwconfig to configure wireless-specific parameters


my ouput:
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host noprefixroute 
       valid_lft forever preferred_lft forever
2: ens4: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1460 qdisc mq state UP group default qlen 1000
    link/ether 42:01:0a:80:00:02 brd ff:ff:ff:ff:ff:ff
    altname enp0s4
    inet 10.128.0.2/32 metric 100 scope global dynamic ens4
       valid_lft 1873sec preferred_lft 1873sec
    inet6 fe80::4001:aff:fe80:2/64 scope link 
       valid_lft forever preferred_lft forever


----
### Commands

0. ifconfig -a: A legacy command still widely used to show all available network interfaces ** 

1. ip link: Used to view and change the status (UP/DOWN) of a specific interface

2. ip addr (or ip a): The modern standard for displaying all interfaces and their assigned IP addresses

3. ip route


When to use which

ip link → is interface up/down?
ip addr → what IPs are assigned?
ip route → where does traffic go? what is default gateway?
ifconfig -a → older all-in-one quick check, especially counters/errors

others:
ifplugstatus: A helpful utility to check if a physical link beat is detected (e.g., if a cable is actually plugged in)
ethtool: Used to query or control the hardware settings and network drivers of an interface

----

Interface,Type,Common Use Case
lo,Virtual / Loopback,"Internal communication (e.g., Nginx talking to Flask on 127.0.0.1)."
ens4,Physical / Virtual Ethernet,"The ""front door."" Communicating with the VPC, Internet, or other VMs."
docker0,Virtual Bridge,(If you had Docker) Used to bridge traffic between containers.

----

What is ens4?
On your Google Cloud VM, ens4 is your Primary Ethernet Interface.

While older Linux systems used eth0, modern distributions (like the Debian 12 you are using) use "Predictable Interface Names." Here is the breakdown of what ens4 specifically tells a sysadmin:

en: Stands for Ethernet.

s: Stands for Hotplug Slot. This indicates the interface is connected via a PCI Express hotplug slot.

4: The specific index or slot number assigned by the system (in GCP, this is the standard naming for the first interface).

----



ip addr



1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host noprefixroute 
       valid_lft forever preferred_lft forever

       > 127.0.0.1/8------------------------------> /8 means network mask 255.0.0.0
        Loopback address
        Only works inside the same machine
        Traffic never leaves the server
        Used for local testing

2: ens4: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1460 qdisc mq state UP group default qlen 1000
    link/ether 42:01:0a:80:00:02 brd ff:ff:ff:ff:ff:ff
    altname enp0s4
    inet 10.128.0.2/32 metric 100 scope global dynamic ens4
       valid_lft 2488sec preferred_lft 2488sec
    inet6 fe80::4001:aff:fe80:2/64 scope link 
       valid_lft forever preferred_lft forever

        > 10.128.0.2/32---------------------------> 32 means network mask 255.255.255.255
        Private/internal cloud network IP
        Used for communication with other systems over the network
        Reachable inside the VPC/subnet rules
        This is the VM’s actual network identity in GCP

**Loopback (`lo`)**

* It is the machine talking to itself.
* `127.0.0.1` means “this same server.”
* Used to test local services without using the real network.

**In your output**

* `lo` = local-only interface
* `ens4` = real network interface connected to the cloud network

**Important things to know about interfaces for admin/interview**

1. **Interface name**

* Example: `lo`, `ens4`
* Tells you which NIC you are dealing with

2. **State**

* `UP` means enabled
* `LOWER_UP` usually means link is actually working underneath too

3. **IP address + CIDR**

* Example: `10.128.0.2/32`
* IP = host address
* `/32` = very specific route, common in cloud setups

4. **MAC address**

* `link/ether 42:01:0a:80:00:02`
* Layer 2 identity of the interface

5. **MTU**

* `1460`
* Max packet size
* Important for fragmentation / VPN / cloud networking issues

6. **IPv4 vs IPv6**

* `inet` = IPv4
* `inet6` = IPv6
* Admin should notice both

7. **Scope**

* `scope host` = only local machine
* `scope global` = usable on network
* `scope link` = only local link/segment

8. **Altname**

* `altname enp0s4`
* Another predictable interface naming reference

**Good interview/admin points**

* Interface can be **up but app still unreachable** → then check route, firewall, listening port
* `ip addr` tells **addressing**, not full connectivity
* After interfaces, usually check:

  * `ip route`
  * `ss -tulpn`
  * `ping` / `curl`
  * firewall rules

**One strong interview line**

* “I use `ip addr` to verify interface state, assigned IPs, MTU, and whether I’m dealing with loopback, link-local, or globally reachable addresses.”


ip route

manojcloudvm@instance-20260404-024439:~$ ip route
default via 10.128.0.1 dev ens4 proto dhcp src 10.128.0.2 metric 100 
10.128.0.1 dev ens4 proto dhcp scope link src 10.128.0.2 metric 100 
169.254.169.254 via 10.128.0.1 dev ens4 proto dhcp src 10.128.0.2 metric 100 

manojcloudvm@instance-20260404-024439:~$ ip route
-------->default via 10.128.0.1<--------- dev ens4 proto dhcp ------------>src 10.128.0.2<----------- metric 100 
-------->10.128.0.1 dev ens4<------------ proto dhcp scope link<------> src 10.128.0.2<--------------- metric 100 
-------->169.254.169.254 via 10.128.0.1 dev ens4<------------------- proto dhcp src 10.128.0.2 metric 100 




**What `ip route` shows**

* It shows the server’s **routing table**
* Meaning: “where should traffic go?”

**Your entries**

`default via 10.128.0.1 dev ens4 ...`

* Default route = path used when destination is not on a more specific known route
* `10.128.0.1` is usually the **gateway** for your VM
* In GCP, this is the next hop/router for your subnet/VPC traffic

`10.128.0.1 dev ens4 ... scope link ...`

* Says gateway `10.128.0.1` itself is directly reachable on interface `ens4`

`169.254.169.254 via 10.128.0.1 dev ens4 ...`

* This is the **cloud metadata server**
* In GCP, VMs use it to get instance metadata, service account tokens, startup script info, etc.

**So: what is `10.128.0.1`?**

* Your VM’s **default gateway / next hop router**
* Your VM sends outbound traffic there

**What is `169.254.169.254`?**

* A special **link-local metadata IP**
* Very common in cloud environments
* Not a normal internet host

**Big picture of this output**
It tells you:

* which interface is used: `ens4`
* what source IP the VM prefers: `10.128.0.2`
* where unknown traffic goes: `10.128.0.1`
* special route to metadata service: `169.254.169.254`

**Admin/interview takeaway**

* `ip addr` = “what IPs do I have?”
* `ip route` = “where will my traffic go?”

**Strong interview line**

* “I check `ip route` to verify the default gateway, source IP selection, and any special routes such as cloud metadata endpoints.”


IMPORTANT


**If interface is down**
Temporary:

```bash
sudo ip link set ens4 up
```

Older way:

```bash
sudo ifconfig ens4 up
```

**Important**

* This may bring interface up only **until reboot**
* If it keeps coming back down, check:

  * network config
  * DHCP/static config
  * cloud/VM NIC attachment
  * NetworkManager/systemd-networkd/netplan

---

**If route is wrong**
First check current routes:

```bash
ip route
```

Delete wrong route:

```bash
sudo ip route del <wrong-route>
```

Add correct route:

```bash
sudo ip route add <network>/<cidr> via <gateway> dev <interface>
```

Example:

```bash
sudo ip route add 192.168.1.0/24 via 10.128.0.1 dev ens4
```

Change default route:

```bash
sudo ip route replace default via 10.128.0.1 dev ens4
```

---

**Important interview point**

* `ip link set ... up` and `ip route add/replace ...` are usually **runtime changes**
* For **persistent** fix, update the OS network config:

  * netplan
  * `/etc/network/interfaces`
  * NetworkManager
  * cloud-init
  * systemd-networkd

**Good interview line**

* “I’d first make the runtime fix to restore connectivity, then update the persistent network configuration so the change survives reboot.”
